# Entrenamiento de YOLOv8 para Frutas (Detección + Clasificación)

Este notebook prepara automáticamente el dataset que descargaste en la carpeta `Fruits` y lanza el entrenamiento de un modelo YOLOv8 unificado.

## 1. Instalación de dependencias
Asegúrate de tener instalada la librería de ultralytics.

In [ ]:
%pip install ultralytics PyYAML

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Generación automática del archivo `data.yaml`
YOLO requiere un archivo YAML con las rutas absolutas a las imágenes y los nombres de las clases. Este script lo genera leyendo tu archivo `classes.txt`.

In [2]:
import os
import yaml
from pathlib import Path

# Rutas absolutas a tu dataset
PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "Fruits"

classes_file = DATASET_DIR / "labels" / "train" / "classes.txt"

if not classes_file.exists():
    print(f"Error: No se encontró el archivo {classes_file}")
else:
    # Leer clases
    with open(classes_file, 'r', encoding='utf-8') as f:
        classes = [line.strip() for line in f.readlines() if line.strip()]
    
    # Crear estructura del YAML
    data_yaml = {
        'path': str(DATASET_DIR.resolve()).replace('\\', '/'),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',  # opcional
        'nc': len(classes),
        'names': classes
    }
    
    # Guardar data.yaml
    yaml_path = DATASET_DIR / "data.yaml"
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_yaml, f, sort_keys=False)
    
    print(f"¡data.yaml generado exitosamente en {yaml_path}!")
    print(f"Clases encontradas ({len(classes)}): {classes}")

¡data.yaml generado exitosamente en c:\Users\jmari\Documents\Universidad\Septimo semestre\Inteligencia artificial\proyecto\ComputerVisionFruits\Fruits\data.yaml!
Clases encontradas (15): ['cucumber', 'apple', 'kiwi', 'banana', 'orange', 'coconut', 'peach', 'cherry', 'pear', 'pomegranate', 'pineapple', 'watermelon', 'melon', 'grape', 'strawberry']


## 3. Entrenamiento del Modelo
Lanzamos el entrenamiento de YOLOv8 usando el modelo base `yolov8n.pt` (recomendado para balance entre velocidad y precisión).

In [3]:
from ultralytics import YOLO
import torch

# Verificar si hay GPU disponible
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Entrenando usando: {'GPU' if device == 0 else 'CPU'}")

# Cargar el modelo base
model = YOLO('yolov8n.pt')

# Iniciar el entrenamiento
# Ajusta los epochs (recomendado: 50 a 100 para un buen resultado)
results = model.train(
    data=str(DATASET_DIR / "data.yaml"),
    epochs=50,             # Número de iteraciones completas sobre el dataset
    imgsz=640,             # Tamaño de imagen (640 es estándar)
    batch=16,              # Tamaño de lote
    device=device,         # 0 para GPU, 'cpu' para CPU
    name='frutas_unificado' # Nombre de la carpeta donde se guardará
)

print("\n=== Entrenamiento Finalizado ===")
print("Tu nuevo modelo entrenado está guardado en:")
print("runs/detect/frutas_unificado/weights/best.pt")

KeyboardInterrupt: 

## 4. Validación rápida
Evalúa qué tan bien quedó el modelo en el conjunto de validación.

In [ ]:
# Validar el modelo recién entrenado
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"mAP50: {metrics.box.map50:.3f}")